# Open Notebook in Colab

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/text_descriptives_tutorial.ipynb)


# Text descriptives tutorial

Explore the length, vocabulary, readability, and sentence patterns of your documents before analysis. This workflow runs locally and makes **no LLM calls**. It does not need an API key.

Run the cells from top to bottom. The default uses the bundled practice CSV. You can also use your own CSV (one document per row) or a folder of `.txt` files. The default English spaCy model is appropriate for the bundled example; use a matching model for other languages.


## Setup (once)

Run the next cell to download LLM Tracker and the practice files if needed, install LLM Tracker and TextDescriptives, and download the English spaCy model. You do not need to clone or download the repository beforehand. Setup may take a few minutes and makes no LLM calls.

This works in Google Colab or a local Jupyter notebook using Python 3.11–3.13. If asked to restart the session/kernel, restart it and rerun this setup cell.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlopen
from shutil import copyfileobj
from zipfile import ZipFile

project_dir = Path.cwd()
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = project_dir.parent

if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = Path.cwd() / "llm_tracker-main"
    if not (project_dir / "src" / "llm_tracker").is_dir():
        print("Downloading LLM Tracker and practice files...")
        with tempfile.TemporaryDirectory() as folder:
            archive = Path(folder) / "llm_tracker.zip"
            url = "https://codeload.github.com/childmindresearch/llm_tracker/zip/refs/heads/main"
            with urlopen(url, timeout=30) as response:
                with archive.open("wb") as destination:
                    copyfileobj(response, destination)
            with ZipFile(archive) as downloaded:
                downloaded.extractall(project_dir.parent)

# Fix the package's current NumPy restriction.
pyproject = project_dir / "pyproject.toml"
settings = pyproject.read_text()
settings = settings.replace('numpy = "^1.26"', 'numpy = ">=1.26,<3"')
pyproject.write_text(settings)

td_version = "2.8.2" if sys.version_info >= (3, 13) else "2.8.4"
print("Installing LLM Tracker and text-descriptive tools...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--only-binary=numpy,pandas,scikit-learn,spacy,thinc,blis",
        "--timeout", "30", "--retries", "1",
        "-e", str(project_dir),
        f"textdescriptives=={td_version}",
        "spacy>=3.8,<3.9",
    ],
    check=True,
    timeout=300,
)

sys.path.insert(0, str(project_dir / "src"))
print("Installing the English language model...")
subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    check=True,
    timeout=180,
)

import llm_tracker
import textdescriptives

print("Setup complete. Continue to Input files.")


## 1. Choose documents

The default practice file is a CSV with one document per row and a `text` column. For your own data, set `input_source` to:

- `"path"` and update `input_data` to a CSV or a folder of `.txt` files available to this Python session.
- `"upload"` to select a CSV or one or more `.txt` files in Colab.
- `"drive"` and set `drive_input_data` to a path under `/content/drive/MyDrive/`.

Colab cannot access files on your computer by local path. For persistent output in Colab, set `output_root` to a folder under `/content/drive/MyDrive/`.


In [ ]:
sample_dir = project_dir / "sample_data" / "tutorial_practice"
input_source = "demo"  # "demo", "path", "upload", or "drive".
input_data = sample_dir / "documents.csv"  # CSV or a folder of .txt files for "path".
drive_input_data = Path("/content/drive/MyDrive/study/documents.csv")
text_column = "text"
output_root = project_dir / "research_results"
spacy_model = "en_core_web_sm"


In [ ]:
if input_source == "demo":
    source_path = sample_dir / "documents.csv"
elif input_source == "path":
    source_path = Path(input_data).expanduser().resolve()
elif input_source == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    source_path = Path(drive_input_data).expanduser().resolve()
    if not source_path.is_relative_to(Path("/content/drive/MyDrive")):
        raise ValueError("Set drive_input_data to a path under /content/drive/MyDrive/.")
elif input_source == "upload":
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No documents selected.")
    text_files = all(Path(name).suffix.lower() == ".txt" for name in uploaded)
    if not text_files and len(uploaded) != 1:
        raise ValueError("Upload one CSV file or only TXT files.")
    upload_dir = Path(tempfile.mkdtemp(prefix="llm_tracker_text_descriptives_"))
    for name, contents in uploaded.items():
        (upload_dir / Path(name).name).write_bytes(contents)
    source_path = upload_dir if text_files else upload_dir / Path(next(iter(uploaded))).name
else:
    raise ValueError('Choose input_source = "demo", "path", "upload", or "drive".')

if not source_path.exists():
    raise FileNotFoundError(f"Documents not found: {source_path}")
if source_path.is_dir():
    from llm_tracker.file_handlers import get_document_files, load_document

    document_paths = get_document_files(source_path)
    if not document_paths:
        raise ValueError(f"No supported TXT documents found in {source_path}.")
    documents = [load_document(path)[0] for path in document_paths]
else:
    import pandas as pd

    input_frame = pd.read_csv(source_path)
    if text_column not in input_frame:
        raise ValueError(f"Text column '{text_column}' was not found in {source_path}.")
    if input_frame.empty:
        raise ValueError(f"No documents found in {source_path}.")
    if input_frame[text_column].isna().any():
        raise ValueError(f"The '{text_column}' column contains missing document text.")
    documents = input_frame[text_column].astype(str).tolist()

print(f"Loaded {len(documents)} documents from {source_path}")


## 2. Summarize the corpus

The summary table reports corpus-level statistics. A timestamped CSV with measurements for each document is also saved under `output_root`.


In [ ]:
from IPython.display import display
from llm_tracker.corpus_summary import summarize_corpus

output_root = Path(output_root).expanduser().resolve()
text_summary, text_by_document = summarize_corpus(
    documents,
    spacy_model=spacy_model,
    include_readability=True,
    include_sentence_complexity=True,
    include_coherence_sbert=False,
    output_dir=output_root,
    input_path=str(source_path),
)
display(text_summary)
